# Task 1: Data Quality and Preparation

Checks aligned to the TfSE brief. Outputs are kept technical; notes only interpret what the checks mean.

## 1. Load and structure

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
files = list((root / "data").glob("*.xlsx"))
DATA_FILE = files[0]

preview = pd.read_excel(DATA_FILE, sheet_name="All trips", header=None, nrows=20)
header = preview.apply(lambda r: r.astype(str).str.strip().eq("caseid").any(), axis=1).idxmax()

trips = pd.read_excel(DATA_FILE, sheet_name="All trips", header=header)
trips = trips.dropna(how="all").dropna(axis=1, how="all").copy()

pd.DataFrame({
    "Metric": ["Trips", "Respondents", "Variables"],
    "Value": [len(trips), trips["caseid"].nunique(), trips.shape[1]]
})

,Metric,Value
0,Trips,6631
1,Respondents,4097
2,Variables,17


**Interpretation:** One row is treated as one recorded trip. `caseid` identifies respondents.

## 2. Missingness and basic consistency

In [2]:
date = pd.to_datetime(trips["Date"], errors="coerce")
start_hour = pd.to_numeric(trips["Start time - first hour"], errors="coerce")
distance_km = pd.to_numeric(trips["Distance (km)"], errors="coerce")
travel_time = pd.to_numeric(trips["Travel Time (minutes)"], errors="coerce")
distance_cat = pd.to_numeric(trips["Distance category"], errors="coerce")
rounded_miles = pd.to_numeric(trips["Distance (miles) - rounded"], errors="coerce")
weight = pd.to_numeric(trips["Weight_2"], errors="coerce")

missing = pd.DataFrame({
    "Variable": trips.columns,
    "Missing": trips.isna().sum().values
})
missing["Missing_%"] = (missing["Missing"] / len(trips) * 100).round(1)

missing.query("Missing > 0").sort_values("Missing", ascending=False)

,Variable,Missing,Missing_%
8,Travel Time (minutes),534,8.1
7,Distance (km),458,6.9
11,Origin eastings,458,6.9
14,Destination northings,458,6.9
13,Destination eastings,458,6.9
12,Origin northings,458,6.9
5,Start time - first hour,293,4.4
4,Day of week,26,0.4
3,Date,26,0.4
6,Main mode,14,0.2


In [3]:
origin_missing = trips[["Origin eastings", "Origin northings"]].isna().any(axis=1)
dest_missing = trips[["Destination eastings", "Destination northings"]].isna().any(axis=1)

co_missing = pd.DataFrame({
    "Check": [
        "Missing distance",
        "Missing travel time",
        "Missing OD coordinates",
        "Missing distance + OD",
        "Missing distance + OD + travel time"
    ],
    "Records": [
        distance_km.isna().sum(),
        travel_time.isna().sum(),
        (origin_missing | dest_missing).sum(),
        (distance_km.isna() & (origin_missing | dest_missing)).sum(),
        (distance_km.isna() & (origin_missing | dest_missing) & travel_time.isna()).sum()
    ]
})
co_missing

,Check,Records
0,Missing distance,458
1,Missing travel time,534
2,Missing OD coordinates,458
3,Missing distance + OD,458
4,Missing distance + OD + travel time,458


**Interpretation:** Co-missingness helps distinguish isolated missing values from records affected by the same underlying mapping or processing issue.

## 3. Duplicates and cross-field checks

In [4]:
exact_dup = trips.duplicated(keep=False)
key_dup = trips.duplicated(["caseid", "Trip number"], keep=False)

dup_fields = [
    "caseid", "Destination eastings", "Destination northings",
    "Start time - first hour", "Travel Time (minutes)", "Main mode"
]
complete_dup = trips[dup_fields].notna().all(axis=1)
potential_dup = complete_dup & trips.duplicated(dup_fields, keep=False)

supplied_day = trips["Day of week"].astype("string").str.strip().str[:3].str.lower()
derived_day = date.dt.day_name().astype("string").str[:3].str.lower()
day_mismatch = date.notna() & trips["Day of week"].notna() & supplied_day.ne(derived_day)

pd.DataFrame({
    "Check": [
        "Exact duplicate rows",
        "Duplicate caseid + Trip number",
        "Potential duplicate records",
        "Date / day mismatch",
        "Start hour outside 0-23"
    ],
    "Records": [
        exact_dup.sum(),
        key_dup.sum(),
        potential_dup.sum(),
        day_mismatch.sum(),
        (start_hour.notna() & ~start_hour.between(0, 23)).sum()
    ]
})

,Check,Records
0,Exact duplicate rows,0
1,Duplicate caseid + Trip number,0
2,Potential duplicate records,12
3,Date / day mismatch,0
4,Start hour outside 0-23,0


**Interpretation:** Potential duplicates are flagged only where all comparison fields are present. They are review flags and are not removed automatically.

## 4. Distance consistency and unusual records

In [5]:
distance_miles = distance_km / 1.609344

zero_distance = distance_km.eq(0)
same_od = (
    trips["Origin eastings"].eq(trips["Destination eastings"])
    & trips["Origin northings"].eq(trips["Destination northings"])
)

cat0 = distance_cat.eq(0)
pd.DataFrame({
    "Check": [
        "Distance category 0",
        "Category 0 + missing distance",
        "Category 0 + zero distance",
        "Zero distance + identical OD"
    ],
    "Records": [
        cat0.sum(),
        (cat0 & distance_km.isna()).sum(),
        (cat0 & zero_distance).sum(),
        (zero_distance & same_od).sum()
    ]
})

,Check,Records
0,Distance category 0,760
1,Category 0 + missing distance,458
2,Category 0 + zero distance,302
3,Zero distance + identical OD,302


In [6]:
bins = [0, 1, 2, 5, 10, 25, 50, 100, np.inf]
labels = range(1, 9)

derived_cat = pd.Series(np.nan, index=trips.index)
positive = distance_miles.gt(0)
derived_cat.loc[positive] = pd.cut(
    distance_miles.loc[positive], bins=bins, labels=labels, right=False
).astype(float)

cat_mismatch = (
    positive & distance_cat.isin(labels)
    & derived_cat.notna() & distance_cat.ne(derived_cat)
)

rounding_boundary = cat_mismatch & rounded_miles.eq(
    np.select(
        [distance_miles.lt(1), distance_miles.lt(2), distance_miles.lt(5),
         distance_miles.lt(10), distance_miles.lt(25), distance_miles.lt(50),
         distance_miles.lt(100)],
        [1, 2, 5, 10, 25, 50, 100],
        default=np.nan
    )
)

trips.loc[cat_mismatch, [
    "Distance (km)", "Distance (miles) - rounded", "Distance category"
]].assign(derived_category=derived_cat.loc[cat_mismatch])

,Distance (km),Distance (miles) - rounded,Distance category,derived_category
872,16.018724,10.0,5,4.0
2000,16.038569,10.0,5,4.0
2791,16.035171,10.0,5,4.0
3158,16.067857,10.0,5,4.0
4300,16.074929,10.0,5,4.0


**Interpretation:** Review boundary mismatches before treating them as data errors; rounded miles can move a trip into the next published band.

In [7]:
MODE_MAX_KM = {
    "Walking": 20, "Mobility scooter": 50, "Pedal cycle": 250,
    "Electric cycle (e-bike)": 250, "Hire e-bike/ e-scooter": 250,
    "Car/ van (as a passenger)": 300, "Car/ van (as the driver)": 300,
    "Rail": 300, "Motorcycle/ moped": 300,
    "Private bus/ coach (e.g. school service, other private service)": 300,
    "Public bus service": 300, "Other coach (e.g. long distance coaches)": 350,
    "Ferry": 350
}

mode_limit = trips["Main mode"].map(MODE_MAX_KM)
mode_distance_flag = distance_km.gt(mode_limit) & mode_limit.notna()

(trips.loc[mode_distance_flag]
 .assign(distance_km=distance_km.loc[mode_distance_flag])
 .groupby("Main mode", observed=False)
 .agg(Trips=("caseid", "size"),
      Respondents=("caseid", "nunique"),
      Min_km=("distance_km", "min"),
      Max_km=("distance_km", "max"))
 .sort_values("Trips", ascending=False)
 .round(1))

,Trips,Respondents,Min_km,Max_km
Main mode,,,,
Walking,121,102,20.0,894.9
Car/ van (as the driver),66,59,301.5,971.3
Car/ van (as a passenger),27,25,301.5,863.4
Rail,21,19,308.8,696.9
Public bus service,11,7,361.6,748.3
Mobility scooter,1,1,65.6,65.6


**Interpretation:** TfSE mode-distance limits are used as plausibility flags for distance analysis, not as automatic deletions.

## 5. Travel-time provenance

In [8]:
travel_source = trips["Travel time source (stated / calculated)"].astype("string").str.strip().str.lower()

source_summary = (
    pd.DataFrame({"Source": travel_source, "Minutes": travel_time})
    .groupby("Source", dropna=False)
    .agg(Trips=("Minutes", "size"),
         Valid_time=("Minutes", lambda x: x.gt(0).sum()),
         Median_minutes=("Minutes", "median"))
)

diag = pd.DataFrame({
    "Mode": trips["Main mode"],
    "Distance_km": distance_km,
    "Minutes": travel_time
}).query("Distance_km > 0 and Minutes > 0").copy()
diag["Min_per_km"] = diag["Minutes"] / diag["Distance_km"]

ratio_summary = (
    diag.groupby("Mode", observed=False)["Min_per_km"]
    .agg(
        Valid_trips="size",
        Median_min_per_km="median",
        Unique_ratios=lambda x: x.round(6).nunique()
    )
    .sort_values("Valid_trips", ascending=False)
    .round(2)
)

display(source_summary.round(2))
display(ratio_summary)

,Trips,Valid_time,Median_minutes
Source,,,
calculated,2,1,897.98
stated,6615,5804,13.31
<NA>,14,0,NaN


,Valid_trips,Median_min_per_km,Unique_ratios
Mode,,,
Car/ van (as the driver),3222,1.52,3081
Walking,1031,12.00,1
Car/ van (as a passenger),746,1.49,735
Public bus service,360,2.40,1
Rail,296,0.86,1
Pedal cycle,63,3.75,1
Taxi/ minicab,60,1.57,59
Electric cycle (e-bike),12,2.73,1
Mobility scooter,8,12.00,1


**Interpretation:** Very low ratio variation within a mode suggests travel time may be partly processed from distance even where the source label says `stated`. Treat travel time cautiously in behavioural interpretation.

## 6. Respondent-trip relationships

In [9]:
trip_num = pd.to_numeric(
    trips["Trip number"].astype("string").str.extract(r"(\d+)", expand=False),
    errors="coerce"
)

trip_counts = trips.groupby("caseid").size()
dates_per_person = trips.assign(_date=date).groupby("caseid")["_date"].nunique()

numbers = trips.assign(_trip=trip_num).groupby("caseid")["_trip"].apply(
    lambda x: sorted(set(x.dropna().astype(int)))
)

internal_gap = numbers.apply(
    lambda x: len(x) > 1 and x != list(range(min(x), max(x) + 1))
)
starts_above_1 = numbers.apply(lambda x: bool(x) and min(x) > 1)

pd.DataFrame({
    "Metric": [
        "Unique respondents",
        "Respondents with >1 trip",
        "Median trips per respondent",
        "Maximum trips per respondent",
        "Respondents spanning >1 survey date",
        "Respondents with internal trip-number gaps",
        "Respondents starting above Trip 1"
    ],
    "Value": [
        trips["caseid"].nunique(),
        (trip_counts > 1).sum(),
        trip_counts.median(),
        trip_counts.max(),
        (dates_per_person > 1).sum(),
        internal_gap.sum(),
        starts_above_1.sum()
    ]
})

,Metric,Value
0,Unique respondents,4097.0
1,Respondents with >1 trip,1472.0
2,Median trips per respondent,1.0
3,Maximum trips per respondent,6.0
4,Respondents spanning >1 survey date,0.0
5,Respondents with internal trip-number gaps,28.0
6,Respondents starting above Trip 1,74.0


**Interpretation:** Trip numbering is analysed within respondent-day records. Numbering gaps are separated from respondents whose observed extract starts after Trip 1.

## 7. Trip-sequence validation

In [10]:
seq = pd.DataFrame({
    "caseid": trips["caseid"],
    "trip": trip_num,
    "hour": start_hour,
    "oe": pd.to_numeric(trips["Origin eastings"], errors="coerce"),
    "on": pd.to_numeric(trips["Origin northings"], errors="coerce"),
    "de": pd.to_numeric(trips["Destination eastings"], errors="coerce"),
    "dn": pd.to_numeric(trips["Destination northings"], errors="coerce")
}).sort_values(["caseid", "trip"])

for c in ["trip", "hour", "oe", "on", "de", "dn"]:
    seq[f"next_{c}"] = seq.groupby("caseid")[c].shift(-1)

seq["consecutive"] = seq["next_trip"].eq(seq["trip"] + 1)
seq["dest_next_origin_m"] = np.hypot(seq["de"] - seq["next_oe"], seq["dn"] - seq["next_on"])
seq["origin_next_origin_m"] = np.hypot(seq["oe"] - seq["next_oe"], seq["on"] - seq["next_on"])
seq["origin_next_dest_m"] = np.hypot(seq["oe"] - seq["next_de"], seq["on"] - seq["next_dn"])

time_ok = seq["consecutive"] & seq[["hour", "next_hour"]].notna().all(axis=1)
od_ok = seq["consecutive"] & seq[["de", "dn", "next_oe", "next_on"]].notna().all(axis=1)

continuous = od_ok & seq["dest_next_origin_m"].le(100)
reverse = continuous & seq["origin_next_dest_m"].le(100)
same_origin = (
    seq["consecutive"]
    & seq[["oe", "on", "next_oe", "next_on"]].notna().all(axis=1)
    & ~continuous
    & seq["origin_next_origin_m"].le(100)
)

pd.DataFrame({
    "Metric": [
        "Consecutive trip pairs",
        "Pairs with usable time",
        "Time non-decreasing (%)",
        "Pairs with usable OD",
        "Spatially continuous (%)",
        "Reverse OD return (%)",
        "Same-origin repeat (%)"
    ],
    "Value": [
        seq["consecutive"].sum(),
        time_ok.sum(),
        round((seq.loc[time_ok, "next_hour"] >= seq.loc[time_ok, "hour"]).mean() * 100, 1),
        od_ok.sum(),
        round(continuous.loc[od_ok].mean() * 100, 1),
        round(reverse.loc[od_ok].mean() * 100, 1),
        round(same_origin.loc[od_ok].mean() * 100, 1)
    ]
})

,Metric,Value
0,Consecutive trip pairs,2506.0
1,Pairs with usable time,2438.0
2,Time non-decreasing (%),88.9
3,Pairs with usable OD,2221.0
4,Spatially continuous (%),49.1
5,Reverse OD return (%),5.9
6,Same-origin repeat (%),50.9


In [11]:
examples = trips[trips["caseid"].isin([6433611624, 6409985436])][
    ["caseid", "Trip number", "Start time - first hour",'Date', "Purpose_category",
     "Main mode", "Origin eastings", "Origin northings",
     "Destination eastings", "Destination northings"]
].sort_values(["caseid", "Trip number"])

examples

,caseid,Trip number,Start time - first hour,Date,Purpose_category,Main mode,Origin eastings,Origin northings,Destination eastings,Destination northings
7,6409985436,$trip1pipe,10.0,2024-11-19,personal business,Car/ van (as the driver),445796.0,130343.0,447107.0,129366.0
8,6409985436,$trip2pipe,12.0,2024-11-19,leisure,Car/ van (as the driver),445796.0,130343.0,448235.0,131151.0
9,6409985436,$trip3pipe,15.0,2024-11-19,personal business,Car/ van (as the driver),445796.0,130343.0,446762.0,130884.0
10,6409985436,$trip4pipe,17.0,2024-11-19,other,Car/ van (as the driver),445796.0,130343.0,446849.0,132053.0
11,6409985436,$trip5pipe,19.0,2024-11-19,leisure,Car/ van (as the driver),445796.0,130343.0,446372.0,131193.0
468,6433611624,$trip1pipe,8.0,2024-11-18,personal business,Pedal cycle,464863.0,98439.0,466581.0,99452.0
470,6433611624,$trip2pipe,11.0,2024-11-18,leisure,Car/ van (as the driver),466581.0,99452.0,465223.0,98688.0
471,6433611624,$trip3pipe,12.0,2024-11-18,leisure,Car/ van (as the driver),465223.0,98688.0,463214.0,99420.0
472,6433611624,$trip4pipe,14.0,2024-11-18,leisure,Car/ van (as the driver),463214.0,99420.0,465223.0,98688.0
469,6433611624,$trip5pipe,15.0,2024-11-18,personal business,Car/ van (as the driver),465223.0,98688.0,466581.0,99452.0


**Interpretation:** `Trip number` provides useful ordering, but a continuous journey chain is inferred only when consecutive OD locations support it.

## 8. Weighting

In [12]:
weight_consistency = (
    trips.assign(_weight=weight)
    .groupby("caseid")["_weight"]
    .nunique(dropna=True)
)

mode = trips["Main mode"].notna()
unweighted = trips.loc[mode, "Main mode"].value_counts(normalize=True).mul(100)
weighted = (
    trips.loc[mode].assign(_w=weight.loc[mode])
    .groupby("Main mode")["_w"].sum()
)
weighted = weighted / weighted.sum() * 100

weight_summary = pd.DataFrame({
    "Metric": [
        "Missing weights", "Non-positive weights",
        "Minimum", "Median", "Maximum",
        "Respondents with inconsistent weights",
        "Maximum mode-share change (pp)"
    ],
    "Value": [
        weight.isna().sum(), weight.le(0).sum(),
        weight.min(), weight.median(), weight.max(),
        weight_consistency.gt(1).sum(),
        (weighted - unweighted).abs().max()
    ]
})

weight_summary.round(3)

,Metric,Value
0,Missing weights,0.000
1,Non-positive weights,0.000
2,Minimum,0.254
3,Median,1.056
4,Maximum,2.942
5,Respondents with inconsistent weights,0.000
6,Maximum mode-share change (pp),1.022


**Interpretation:** Use `Weight_2` for headline behavioural proportions; retain raw counts for sample size and QA.

## 9. Geographic and destination quality

In [13]:
oe = pd.to_numeric(trips["Origin eastings"], errors="coerce")
on = pd.to_numeric(trips["Origin northings"], errors="coerce")
de = pd.to_numeric(trips["Destination eastings"], errors="coerce")
dn = pd.to_numeric(trips["Destination northings"], errors="coerce")

valid_origin = oe.notna() & on.notna()
valid_dest = de.notna() & dn.notna()
valid_od = valid_origin & valid_dest

pd.DataFrame({
    "Check": [
        "Valid origin pairs",
        "Valid destination pairs",
        "Complete OD pairs",
        "Origin within broad BNG range",
        "Destination within broad BNG range",
        "Identical OD coordinates"
    ],
    "Records": [
        valid_origin.sum(),
        valid_dest.sum(),
        valid_od.sum(),
        (valid_origin & oe.between(0, 700000) & on.between(0, 1300000)).sum(),
        (valid_dest & de.between(0, 700000) & dn.between(0, 1300000)).sum(),
        (valid_od & oe.eq(de) & on.eq(dn)).sum()
    ]
})

,Check,Records
0,Valid origin pairs,6173
1,Valid destination pairs,6173
2,Complete OD pairs,6173
3,Origin within broad BNG range,6173
4,Destination within broad BNG range,6173
5,Identical OD coordinates,302


## 10. Spatial readiness

In [14]:
WORKING_CRS = "EPSG:27700"

origin_ll = gpd.GeoSeries(
    gpd.points_from_xy(oe[valid_origin], on[valid_origin]),
    crs=WORKING_CRS
).to_crs(4326)

dest_ll = gpd.GeoSeries(
    gpd.points_from_xy(de[valid_dest], dn[valid_dest]),
    crs=WORKING_CRS
).to_crs(4326)

straight_km = pd.Series(np.nan, index=trips.index)
straight_km.loc[valid_od] = np.hypot(
    de.loc[valid_od] - oe.loc[valid_od],
    dn.loc[valid_od] - on.loc[valid_od]
) / 1000

ratio = distance_km / straight_km
valid_ratio = distance_km.gt(0) & straight_km.gt(0)
upper = ratio.loc[valid_ratio].quantile(0.99)

pd.DataFrame({
    "Check": [
        "Positive reported + straight-line distance",
        "Reported distance below straight-line distance",
        "Ratio above 99th percentile"
    ],
    "Records": [
        valid_ratio.sum(),
        (valid_ratio & ratio.lt(1)).sum(),
        (valid_ratio & ratio.gt(upper)).sum()
    ]
})

,Check,Records
0,Positive reported + straight-line distance,5871
1,Reported distance below straight-line distance,10
2,Ratio above 99th percentile,59


**Interpretation:** EPSG:27700 is a working assumption for spatial QA. Straight-line OD distance is used only as a plausibility check, not as a replacement for reported journey distance.

## 11. Analysis-ready flags

In [15]:
analysis_ready = trips.copy()

analysis_ready["trip_number_numeric"] = trip_num
analysis_ready["potential_duplicate_flag"] = potential_dup
analysis_ready["date_day_mismatch_flag"] = day_mismatch
analysis_ready["distance_category_zero_flag"] = cat0
analysis_ready["distance_category_review_flag"] = cat_mismatch
analysis_ready["zero_distance_flag"] = zero_distance
analysis_ready["mode_distance_review_flag"] = mode_distance_flag
analysis_ready["trip_internal_gap_flag"] = trips["caseid"].isin(internal_gap[internal_gap].index)
analysis_ready["trip_starts_above_1_flag"] = trips["caseid"].isin(starts_above_1[starts_above_1].index)
analysis_ready["valid_distance"] = distance_km.gt(0)
analysis_ready["distance_analysis_valid"] = distance_km.gt(0) & ~mode_distance_flag
analysis_ready["valid_travel_time"] = travel_time.gt(0)
analysis_ready["travel_time_source"] = travel_source
analysis_ready["valid_od_geometry"] = valid_od
analysis_ready["same_origin_destination_flag"] = valid_od & oe.eq(de) & on.eq(dn)
analysis_ready["od_straight_line_km"] = straight_km
analysis_ready["reported_to_straight_line_ratio"] = ratio
analysis_ready["spatial_distance_review_flag"] = valid_ratio & (ratio.lt(1) | ratio.gt(upper))

output_dir = root / "output"
output_dir.mkdir(exist_ok=True)

analysis_ready.to_csv(output_dir / "task1_analysis_ready_trips.csv", index=False)

pd.DataFrame({
    "Area": [
        "Trips", "Respondents", "Exact duplicates",
        "Potential duplicate records", "Mode-distance review flags",
        "Internal trip-number gaps", "Complete OD coordinates",
        "Missing weights", "Distance-analysis valid trips"
    ],
    "Value": [
        len(trips), trips["caseid"].nunique(), exact_dup.sum(),
        potential_dup.sum(), mode_distance_flag.sum(),
        internal_gap.sum(), valid_od.sum(),
        weight.isna().sum(),
        analysis_ready["distance_analysis_valid"].sum()
    ]
})

,Area,Value
0,Trips,6631
1,Respondents,4097
2,Exact duplicates,0
3,Potential duplicate records,12
4,Mode-distance review flags,247
5,Internal trip-number gaps,28
6,Complete OD coordinates,6173
7,Missing weights,0
8,Distance-analysis valid trips,5624


**Interpretation:** The analysis-ready file preserves the original records and adds QA flags so later analyses can apply only the checks relevant to each question.